# Embeddings and Semantic Similarity

This notebook explores how textual information can be represented
as numerical vectors and how those representations can be compared
to measure semantic similarity.

The goal is to understand the foundations behind semantic search
and Retrieval-Augmented Generation (RAG).

In [1]:
!pip install -q sentence-transformers

In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer

In [ ]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)

## Single and Batch Embeddings

### Single Sentence

In [4]:
sentence_simple = "Amazon S3 is an object storage service."

embedding_simple = model.encode(sentence_simple)

print(type(embedding_simple))
print(embedding_simple.shape)
print(embedding_simple[:10])

<class 'numpy.ndarray'>
(384,)
[-0.04856018 -0.06385586 -0.06854329 -0.02536501  0.0656332  -0.02334097
 -0.05347238 -0.05859874  0.08805332  0.05768296]


In [5]:
print(np.linalg.norm(embedding_simple))

1.0


### Batch Encoding

In [6]:
sentences_multiples = [
    "Amazon S3 is an object storage service.",
    "S3 can be used to store files and objects in AWS.",
    "Amazon EC2 provides virtual computing resources.",
    "I cooked pasta yesterday."
]

In [7]:
embedding_multiple = model.encode(sentences_multiples)

In [8]:
print(type(embedding_multiple))
print(embedding_multiple.shape)

<class 'numpy.ndarray'>
(4, 384)


## Cosine similarity

In [9]:
def cosine_similarity_manual(a, b):
    return np.dot(a, b) / (
        np.linalg.norm(a) * np.linalg.norm(b)
    )

In [10]:
sim_s3_s3 = cosine_similarity_manual(
    embedding_multiple[0],
    embedding_multiple[1]
)

sim_s3_ec2 = cosine_similarity_manual(
    embedding_multiple[0],
    embedding_multiple[2]
)

sim_s3_pasta = cosine_similarity_manual(
    embedding_multiple[0],
    embedding_multiple[3]
)

print("S3 vs S3:", sim_s3_s3)
print("S3 vs EC2:", sim_s3_ec2)
print("S3 vs Pasta:", sim_s3_pasta)

S3 vs S3: 0.8390608
S3 vs EC2: 0.39915842
S3 vs Pasta: -0.021304518


### Semantic vs Lexical Similarity

The following example compares sentences that share similar meanings with sentences that merely share vocabulary.

In [11]:
sentences_multiples_lexical = [
    "The bank approved the loan.",
    "The financial institution accepted the credit application.",
    "The bank is next to the supermarket."
]

In [12]:
embedding_multiple_lexical = model.encode(sentences_multiples_lexical)

In [13]:
sim_1_2 = cosine_similarity_manual(
    embedding_multiple_lexical[0],
    embedding_multiple_lexical[1]
)

sim_1_3 = cosine_similarity_manual(
    embedding_multiple_lexical[0],
    embedding_multiple_lexical[2]
)

sim_2_3 = cosine_similarity_manual(
    embedding_multiple_lexical[1],
    embedding_multiple_lexical[2]
)

print("First vs Second:", sim_1_2)
print("First vs Third:", sim_1_3)
print("Second vs Third:", sim_2_3)

First vs Second: 0.5944297
First vs Third: 0.4156158
Second vs Third: 0.2846702


## Semantic Retrieval

In [14]:
documents = [
    "Amazon S3 provides object storage.",
    "Amazon EC2 provides virtual machines.",
    "Amazon RDS provides managed relational databases.",
    "AWS Lambda runs code without managing servers.",
    "Amazon CloudFront is a content delivery network."
]

In [15]:
document_embeddings = model.encode(documents)

print(document_embeddings.shape)

(5, 384)


In [16]:
query = "Where can I store files in AWS?"

query_embedding = model.encode(query)

In [17]:
scores = []

for document, embedding in zip(documents, document_embeddings):
    similarity = cosine_similarity_manual(
        query_embedding,
        embedding
    )

    scores.append((document, similarity))

scores = sorted(
    scores,
    key=lambda x: x[1],
    reverse=True
)

for document, score in scores:
    print(f"{score:.4f} - {document}")

0.5855 - Amazon S3 provides object storage.
0.4409 - Amazon RDS provides managed relational databases.
0.4072 - Amazon CloudFront is a content delivery network.
0.4016 - AWS Lambda runs code without managing servers.
0.3607 - Amazon EC2 provides virtual machines.


### Normalization
The embeddings produced in this example already have unit norm.
Explicit normalization guarantees this property and allows cosine
similarity to be computed directly as a dot product.

In [18]:
normalized_embeddings = model.encode(
    documents,
    normalize_embeddings=True
)

normalized_query = model.encode(
    query,
    normalize_embeddings=True
)

In [19]:
print(np.linalg.norm(normalized_query))
print(np.linalg.norm(normalized_embeddings[0]))

1.0
1.0


### Cosine =  Dot product

In [20]:
cos = cosine_similarity_manual(
    normalized_query,
    normalized_embeddings[0]
)

dot = np.dot(
    normalized_query,
    normalized_embeddings[0]
)

print("Cosine: ",cos)
print("Dot: ",dot)

Cosine:  0.58553827
Dot:  0.58553827


## Key Takeaways

- Embedding models represent text as dense numerical vectors.
- Semantically similar texts tend to have similar vector representations.
- Cosine similarity measures the similarity between vector directions.
- Semantic similarity does not require exact lexical overlap.
- Document embeddings can be ranked against a query embedding to perform semantic retrieval.
- With normalized embeddings, cosine similarity is equivalent to the dot product.